# Data Collection

This notebook collects raw warning verification data from the IEM Convective Outlook Warning (COW) API for 122 NWS Weather Forecast Offices across six calendar years (2020-2025). Data is filtered to three phenomena: Tornado (TO), Severe Thunderstorm (SV), and Flash Flood (FF) warnings. One JSON file is written per WFO-year to `data/raw/COW/`, which is treated as immutable once collected. The collection is idempotent; re-running the notebook skips files already on disk. Project design, data model, and methodological notes are documented in `README.md`.

## Implementation

Collection is implemented across three classes in `src/collection/`:

| Class | Responsibility |
|---|---|
| `WFORegistry` | Loads `wfo_list.csv` and provides the 122 WFO call signs used as API query parameters |
| `COWClient` | Wraps the IEM COW API; handles a single WFO-year fetch with retries and exponential backoff |
| `COWCollector` | Orchestrates the full loop across all WFOs and years; skips files already on disk so the run can be safely interrupted and resumed |

Each WFO-year response is written as a separate JSON file to `data/raw/COW/{WFO}_{YEAR}.json`. Files are never overwritten because the raw directory is treated as immutable once collected.

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from collection import COWClient, COWCollector, WFORegistry

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Paths
WFO_FILE = Path("../data/wfo_list.csv")
COW_DIR  = Path("../data/raw/COW")

# IEM COW API endpoint
COW_BASE_URL = "https://mesonet.agron.iastate.edu/api/1/cow.json"

# VTEC phenomena codes to collect and analyze:
#   TO = Tornado Warning
#   SV = Severe Thunderstorm Warning
#   FF = Flash Flood Warning
PHENOMENA = ["TO", "SV", "FF"]

# Analysis window: 2020–2024 are the pre-treatment baseline years;
# 2025 is the treatment year (post-staffing-cuts).
YEARS = list(range(2020, 2026))

# Seconds to pause between API requests to avoid overwhelming the IEM server
RATE_LIMIT = 0.3

In [2]:
registry  = WFORegistry(WFO_FILE)
client    = COWClient(phenomena=PHENOMENA, base_url=COW_BASE_URL)
collector = COWCollector(registry, client, COW_DIR, years=YEARS, rate_limit=RATE_LIMIT)

print(registry)
print(client)
print(collector)

summary = collector.collect()
print(summary)

14:51:06  INFO     Collection complete - {'wrote': 0, 'skipped': 732, 'failed': 0}


WFORegistry(122 offices from wfo_list.csv)
COWClient(phenomena=['TO', 'SV', 'FF'], url=https://mesonet.agron.iastate.edu/api/1/cow.json)
COWCollector(122 WFOs, years=2020-2025, output=../data/raw/COW)
{'wrote': 0, 'skipped': 732, 'failed': 0}
